# Alonso & Marder (2020), membrane potential distributions, from Dryad m31b1h7

Paper: Alonso LM, Marder E (2020), eLife 9:e55470, doi:10.7554/eLife.55470; claims in
`claims/alonso-2020.yaml` (version of record, main text, legends and Methods).

Dataset: Dryad doi:10.5061/dryad.m31b1h7, version id 94811 (the only published version, so the
one pinned), file `alonso_marder_tempcomp_2019_root.zip` (MD5 from Dryad's file list checked on
load; SHA-256 printed below). The models are simulated with the port in `pyloric.py`, compared
with the deposited Java in `reproduce_alonso_2020.ipynb`.

Method (Materials and methods, Membrane potential and current shares distributions): each
temperature simulated from the deposited initial conditions for 30 s at dt = 0.001 ms; the last
10 s used; 5 x 10^5 samples per distribution, so V is kept every 20 steps (0.02 ms). Histograms
in 0.1 mV bins (a plotting choice).

The Methods use 101 temperatures. At dt = 0.001 ms one temperature costs about 1500 simulated
seconds at dt = 0.05 (50 times the steps), about 23 s of the runner's 4 CPUs, so 101
temperatures would take about 40 min. This notebook uses 26 temperatures, 10 to 25 degrees C in
steps of 0.6 degrees C, for G7P2WE, and 6 (10, 13, 16, 19, 22, 25 degrees C, all on that grid)
for 4X6W66 and MAXTTP.

| Claim | Tested |
|---|---|
| fig2-lp-spike-amplitude-threshold | tested (26 of 101 temperatures) |
| fig2s1-peak-voltage-diverse | tested (6 temperatures) |

In [ ]:
import resource
import time

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import pyloric as P

T_START = time.time()
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)
raw = P.read_zip_bytes()
print(P.digests(raw))
Z = P.open_zip(raw)
MODELS = P.load_models(Z)
CI = P.load_ci(Z)

## Data check and the runs

In [ ]:
print(len(MODELS), "models; used here: G7P2WE, 4X6W66 (4?6W66 of the Methods), MAXTTP")
T26 = np.round(10 + 0.6 * np.arange(26), 1)
T6 = np.array([10.0, 13.0, 16.0, 19.0, 22.0, 25.0])
tasks = [(("G7P2WE", T), MODELS["G7P2WE"], CI, T, 30.0, 10.0, 0.001, 20) for T in T26]
tasks += [((n, T), MODELS[n], CI, T, 30.0, 10.0, 0.001, 20) for n in ("4X6W66", "MAXTTP") for T in T6]
t0 = time.time()
DIST = {r["key"]: r for r in P.pmap(P.run_distribution, tasks)}
print(f"{len(tasks)} runs in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
print("all runs finite:", all(r["finite"] for r in DIST.values()))
print("samples per distribution:", {r["cells"][0]["n"] for r in DIST.values()})
rows = [dict(model=k[0], T=k[1], cell=c["cell"], mean=c["mean"], vmin=c["vmin"], vmax=c["vmax"], spikes=c["spikes"]) for k, r in DIST.items() for c in r["cells"]]
D = pd.DataFrame(rows)
print("V range over all runs: %.1f to %.1f mV" % (D.vmin.min(), D.vmax.max()))
print(D.groupby(["model", "cell"])[["mean", "vmin", "vmax", "spikes"]].agg(["min", "max"]).round(1).to_string())

## fig2-lp-spike-amplitude-threshold

> In the LP cell of the model of Figure 2, spike amplitude decreases and the spike threshold
> depolarizes as temperature rises from 10 to 25 degrees C.

G7P2WE, LP. Amplitude readings: (a) mean spike peak minus the mean trough before each spike
(`pyloric.spike_shapes`); (b) the upper envelope, the highest V in the distribution. Threshold
reading (a): V where dV/dt first exceeds 10 mV/ms after the trough before each spike. Threshold
reading (b), where the distribution's spike region begins, is not computed (see the summary).

In [ ]:
lp = pd.DataFrame([dict(T=T, amp_a=DIST[("G7P2WE", T)]["cells"][1]["amp"], envelope_b=DIST[("G7P2WE", T)]["cells"][1]["vmax"],
                        threshold_a=DIST[("G7P2WE", T)]["cells"][1]["thr"], spikes=DIST[("G7P2WE", T)]["cells"][1]["spikes"],
                        peak=DIST[("G7P2WE", T)]["cells"][1]["peak"], mean_V=DIST[("G7P2WE", T)]["cells"][1]["mean"]) for T in T26]).set_index("T")
print(lp.round(2).to_string())
r_amp_a = stats.spearmanr(lp.index, lp.amp_a)[0]
r_amp_b = stats.spearmanr(lp.index, lp.envelope_b)[0]
r_thr_a = stats.spearmanr(lp.index, lp.threshold_a, nan_policy="omit")[0]
print(f"\nSpearman with temperature: amplitude (a) {r_amp_a:.3f}, upper envelope (b) {r_amp_b:.3f}, threshold (a) {r_thr_a:.3f}")
print(f"amplitude (a) from {lp.amp_a.iloc[0]:.1f} to {lp.amp_a.iloc[-1]:.1f} mV; envelope from {lp.envelope_b.iloc[0]:.1f} to "
      f"{lp.envelope_b.iloc[-1]:.1f} mV; threshold (a) from {lp.threshold_a.iloc[0]:.1f} to {lp.threshold_a.iloc[-1]:.1f} mV")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
edges = -100 + 0.1 * np.arange(1601)
H = np.array([DIST[("G7P2WE", T)]["cells"][1]["hist"] for T in T26]).T
ax[0].imshow(np.log10(H + 1), aspect="auto", origin="lower", extent=[T26[0], T26[-1], -100, 60], cmap="gray_r")
ax[0].set_xlabel("temperature (C)"); ax[0].set_ylabel("V (mV)"); ax[0].set_title("LP, G7P2WE: log10(count + 1)")
ax[1].plot(lp.index, lp.amp_a, "o-", label="amplitude (a)"); ax[1].plot(lp.index, lp.threshold_a, "o-", label="threshold (a)")
ax[1].plot(lp.index, lp.envelope_b, "o-", label="upper envelope (b)"); ax[1].legend(fontsize=7); ax[1].set_xlabel("temperature (C)")
plt.tight_layout(); plt.show()
print("histogram axis -100 to 60 mV; samples outside it:", int(sum(DIST[("G7P2WE", T)]["cells"][1]["n"] - DIST[("G7P2WE", T)]["cells"][1]["hist"].sum() for T in T26)))

**Result: partially reproduced.** In G7P2WE's LP cell the spike amplitude falls from 10 to 25 C
under both readings (peak minus trough 56.9 to 26.2 mV; upper envelope 16.7 to -1.8 mV; Spearman
-1.000 each). The threshold under reading (a) does not depolarize steadily (-27.8 to -27.5 mV,
Spearman -0.197); reading (b) was not computed. 26 of the Methods' 101 temperatures were run.

## fig2s1-peak-voltage-diverse

> The effect of temperature on the membrane potential differs across models; in three example
> models the peak voltage of spikes decreases, increases, or stays relatively constant with
> temperature, and cells can become more depolarized or more hyperpolarized.

G7P2WE (left), 4X6W66 (middle, 4?6W66 of the Methods) and MAXTTP (right); upper envelope (highest
V) of every cell at 10 and 25 degrees C. Decrease or increase: a change of more than 5 mV in that
direction in any cell; constant: within 5 mV in every cell (2 and 10 mV variants).

In [ ]:
env = pd.DataFrame({(n, c): [DIST[(n, T)]["cells"][k]["vmax"] for T in T6] for n in ("G7P2WE", "4X6W66", "MAXTTP")
                    for k, c in enumerate(P.CELLS)}, index=T6)
mv = pd.DataFrame({(n, c): [DIST[(n, T)]["cells"][k]["mean"] for T in T6] for n in ("G7P2WE", "4X6W66", "MAXTTP")
                   for k, c in enumerate(P.CELLS)}, index=T6)
print("upper envelope (mV):"); print(env.round(2).to_string())
print("\nmean V (mV):"); print(mv.round(2).to_string())
chg = env.loc[25.0] - env.loc[10.0]
mchg = mv.loc[25.0] - mv.loc[10.0]
print("\nchange of the upper envelope, 25 minus 10 C (mV):", chg.round(2).to_dict())
print("change of the mean V, 25 minus 10 C (mV):", mchg.round(2).to_dict())
for d in (2.0, 5.0, 10.0):
    dec = bool((chg["G7P2WE"] < -d).any()); inc = bool((chg["4X6W66"] > d).any()); const = bool((chg["MAXTTP"].abs() <= d).all())
    print(f"bound {d:.0f} mV: G7P2WE decreases {dec}; 4X6W66 increases {inc}; MAXTTP constant {const}")
print("mean V rises in", int((mchg > 0).sum()), "cells and falls in", int((mchg < 0).sum()), "of 9")

**Result: reproduced.** From 10 to 25 C the upper envelope falls in every cell of G7P2WE (by 19.1,
18.5 and 13.8 mV), rises in every cell of 4X6W66 (3.9, 7.3 and 6.3 mV; more than 5 mV in LP and
PY), and stays within 5 mV in every cell of MAXTTP (-1.5, -2.5 and 1.0 mV). The 2 mV variant loses
MAXTTP and the 10 mV variant loses 4X6W66. The mean V rises in 8 of the 9 cells and falls in one
(4X6W66 LP, -0.26 mV).

## Summary

| Claim | Outcome |
|---|---|
| fig2-lp-spike-amplitude-threshold | partially reproduced |
| fig2s1-peak-voltage-diverse | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: 26 temperatures (G7P2WE) and 6 (4X6W66, MAXTTP) in
place of 101; spike amplitude and threshold are read as stated in the claims file, threshold
reading (b) not computed. Nothing was found about the dataset beyond the main notebook.

In [ ]:
print(f"notebook run time {(time.time() - T_START) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")